# Navegación por Isobáticas
Simulación táctica de recalada con niebla siguiendo una línea de profundidad (sonda).

Este simulador está diseñado para fines educativos. **No lo utilices para la navegación real.**

<a href="https://colab.research.google.com/github/jorgejuan007/Nautica/blob/main/simulaciones/75_navegacion_isobaticas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

def simulador_isobatica(sonda_objetivo, margen_error):
    # Simula la navegación buscando una línea de profundidad en la niebla
    x = np.linspace(0, 10, 100)
    
    # Perfil del fondo marino (gradiente)
    fondo_real = 5 + (x**2) * 0.2 + np.sin(x*3) * 1.5
    
    # Trayectoria del barco intentando mantener la sonda_objetivo
    # Si sonda > objetivo: acercarse a la costa (caer a estribor)
    # Si sonda < objetivo: alejarse (caer a babor)
    
    trayectoria = np.zeros_like(x)
    rumbo_actual = 0
    y_barco = 3.0 # Empezamos lejos de la isobática
    
    for i in range(len(x)):
        prof_actual = fondo_real[i] - y_barco # muy simplificado
        error = prof_actual - sonda_objetivo
        
        if abs(error) < margen_error:
            rumbo_actual = 0 # Mantener
        elif error > 0:
            rumbo_actual -= 0.1 # Muy profundo, acercar
        else:
            rumbo_actual += 0.2 # Muy somero, alejar rápido!
            
        y_barco += rumbo_actual
        trayectoria[i] = y_barco
        
    fig, ax = plt.subplots(figsize=(10, 4))
    
    # Mapeo de profundidad simulado
    X, Y = np.meshgrid(np.linspace(0, 10, 50), np.linspace(-5, 10, 50))
    Z = 5 + (X**2)*0.2 + np.sin(X*3)*1.5 - Y
    
    c = ax.contourf(X, Y, Z, levels=[0, sonda_objetivo-margen_error, sonda_objetivo+margen_error, 50], colors=['#ff9999', '#99ccff', '#000066'])
    ax.plot(x, trayectoria, 'k-', lw=3, label='Trayectoria del Barco')
    
    ax.set_title(f'Navegación por Isobática de {sonda_objetivo}m')
    ax.legend()
    plt.show()

so = widgets.FloatSlider(value=10, min=2, max=30, description='Sonda Objetivo(m):')
me = widgets.FloatSlider(value=1.0, min=0.1, max=5, description='Margen Tolerancia:')

out = widgets.interactive_output(simulador_isobatica, {'sonda_objetivo': so, 'margen_error': me})
display(widgets.VBox([so, me, out]))
